# ISOM 835 · Advanced Track — Uplift Modeling: Who Is Worth Persuading?
**Optional companion to Session 6 · Prof. Hasan Arslan**

A churn model tells you who is *likely to leave*. It does not tell you who an offer will *persuade*. Uplift modeling estimates the **treatment effect per customer** from an A/B test and targets only the persuadables — the pattern Wayfair, Uber, and every serious marketing science team use.

> **Frame it.** *Unit:* one customer · *Treatment:* a retention offer, randomly assigned in a pilot · *Outcome:* stayed (1) / churned (0) · *Quantity of interest:* uplift = P(stay | offer) − P(stay | no offer) for **this** customer · *Decision:* who gets the offer next quarter.

Four customer types: **persuadables** (stay only if offered), **sure things** (stay anyway), **lost causes** (leave anyway), **sleeping dogs** (leave *because* you contacted them). Only the first group is worth $50.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(835)

## 1. Simulate a pilot with a known truth
Real uplift data is rare and proprietary (Criteo's 25M-row dataset is the public exception). We simulate 20,000 customers from the Telco feature space with a hidden, heterogeneous treatment effect so we can grade ourselves.

In [ ]:
URL = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(URL); df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
X = pd.get_dummies(df[['tenure', 'MonthlyCharges', 'Contract', 'InternetService', 'PaymentMethod', 'TechSupport']], drop_first=True).astype(float)
X = X.sample(20000, replace=True, random_state=835).reset_index(drop=True)

# hidden truth: baseline stay probability + a treatment effect that depends on the customer
z_base = -0.4 + 0.03 * X['tenure'] - 0.012 * X['MonthlyCharges'] + 1.2 * X.get('Contract_Two year', 0) + 0.6 * X.get('Contract_One year', 0)
m2m = (X.get('Contract_Two year', 0) == 0) & (X.get('Contract_One year', 0) == 0)
tau = (0.28 * ((X['tenure'] >= 6) & (X['tenure'] <= 30) & m2m)          # persuadables: settled-in month-to-month customers
       - 0.06 * (X['tenure'] > 48))                                       # sleeping dogs: long-tenure customers react badly to contact
# lost causes: brand-new, high-charge customers — the churn model's favourite target — leave whatever you do (tau = 0 there)
tau = tau.clip(-0.3, 0.4).values; z_base = z_base.values
treated = rng.random(len(X)) < 0.5                                 # randomized pilot: half get the offer
odds_shift = np.log(np.clip(0.5 + tau, 0.05, 0.95) / np.clip(0.5 - tau, 0.05, 0.95))
p_stay = 1 / (1 + np.exp(-(z_base + np.where(treated, odds_shift, 0))))
y = (rng.random(len(X)) < p_stay).astype(int)
print(f'treated {treated.mean():.2f}   stay rate: control {y[~treated].mean():.3f}   treated {y[treated].mean():.3f}   → average uplift {y[treated].mean() - y[~treated].mean():+.3f}')
print(f'hidden truth: {(tau > 0.05).mean():.0%} persuadable, {(tau < -0.02).mean():.0%} sleeping dogs')

## 2. The wrong model: a churn model
Train the usual churn model on the control group and target the riskiest customers. It finds people likely to leave — including lost causes and sleeping dogs.

In [ ]:
idx = np.arange(len(X)); tr, te = train_test_split(idx, test_size=0.4, random_state=835)
churn = HistGradientBoostingClassifier(random_state=835).fit(X.iloc[tr][~treated[tr]], 1 - y[tr][~treated[tr]])
risk = churn.predict_proba(X.iloc[te])[:, 1]                       # P(churn) on the test half

## 3. The right model: a T-learner
Two models — one on treated customers, one on control — and the uplift is the difference of their predictions. (S-learner, X-learner, and uplift trees are the alternatives; CausalML has them all.)

In [ ]:
m_t = HistGradientBoostingClassifier(random_state=835).fit(X.iloc[tr][treated[tr]], y[tr][treated[tr]])
m_c = HistGradientBoostingClassifier(random_state=835).fit(X.iloc[tr][~treated[tr]], y[tr][~treated[tr]])
uplift_hat = m_t.predict_proba(X.iloc[te])[:, 1] - m_c.predict_proba(X.iloc[te])[:, 1]
tau_te = tau[te]
print(f'correlation with hidden truth: uplift model {np.corrcoef(uplift_hat, tau_te)[0,1]:.2f}   churn model {np.corrcoef(risk, tau_te)[0,1]:.2f}')

## 4. Evaluate without the hidden truth: the Qini / uplift curve
In real life you never see τ. You *can* rank the test customers by predicted uplift, take the top k%, and compare stay rates between treated and control **within that slice** — the incremental customers saved per 100 contacted.

In [ ]:
def uplift_curve(score, y_te, t_te, steps=20):
    order = np.argsort(-score); out = []
    for f in np.linspace(0.05, 1, steps):
        top = order[:int(f * len(order))]; yt, tt = y_te[top], t_te[top]
        inc = (yt[tt].mean() - yt[~tt].mean()) * len(top) if tt.sum() and (~tt).sum() else 0   # incremental stays if everyone in slice were treated
        out.append((f, inc))
    return np.array(out)
y_te, t_te = y[te], treated[te]
u_up, u_risk, u_rand = uplift_curve(uplift_hat, y_te, t_te), uplift_curve(risk, y_te, t_te), uplift_curve(rng.random(len(te)), y_te, t_te)
plt.figure(figsize=(7, 4)); plt.plot(u_up[:, 0], u_up[:, 1], color='#2ee6c5', lw=2, label='uplift model (T-learner)'); plt.plot(u_risk[:, 0], u_risk[:, 1], color='#ff6b8b', lw=2, label='churn-risk model'); plt.plot(u_rand[:, 0], u_rand[:, 1], color='gray', ls='--', label='random')
plt.xlabel('share of customers contacted (ranked by model)'); plt.ylabel('incremental customers saved'); plt.legend(); plt.title('Uplift curve: who to contact first'); plt.show()

In [ ]:
# Contact budget: 20% of customers. Who should get the offer?
k = int(0.2 * len(te))
for name, score in [('uplift model', uplift_hat), ('churn-risk model', risk)]:
    top = np.argsort(-score)[:k]; yt, tt = y_te[top], t_te[top]
    inc = (yt[tt].mean() - yt[~tt].mean()) * k
    print(f'{name:17s} top 20%: incremental stays ≈ {inc:6.0f}   (true mean uplift in slice {tau_te[top].mean():+.3f}; sleeping dogs in slice {(tau_te[top] < -0.02).mean():.0%})')

The churn model fills its top slice with the riskiest customers — brand-new, high-charge accounts that leave **whatever you do** (lost causes) — so the offers buy little. The uplift model finds the settled-in month-to-month customers who *can* be kept, and it avoids the long-tenure sleeping dogs the churn model would never flag but a blanket campaign would poke.

## 5. With a library: CausalML (optional)

In [ ]:
# OPTIONAL — pip install causalml   (Colab: !pip install -q causalml)
try:
    from causalml.inference.meta import BaseXClassifier
    from lightgbm import LGBMClassifier
    xl = BaseXClassifier(outcome_learner=LGBMClassifier(verbose=-1), effect_learner=__import__('lightgbm').LGBMRegressor(verbose=-1))
    xl.fit(X.iloc[tr].values, treated[tr].astype(int), y[tr])
    cate = xl.predict(X.iloc[te].values).ravel()
    print(f'X-learner correlation with hidden truth: {np.corrcoef(cate, tau_te)[0,1]:.2f}')
except Exception as e:
    print('causalml not available →', type(e).__name__, '(the T-learner above is the same idea in pure scikit-learn)')

## 6. Your turn
1. **Budget sweep.** Repeat the 20% comparison at 5%, 10%, 40%. Where does the churn model do the most damage?
2. **S-learner.** Fit one model with `treated` as a feature and compute uplift as the prediction difference with the flag flipped. Does it find the sleeping dogs?
3. **Business memo.** In three sentences: what would you tell a marketing director who wants to "send the offer to everyone likely to churn"?

## What this adds to Session 6
- A churn score ranks by **risk**; an uplift score ranks by **persuadability** — and only the second one is what an offer buys.
- Uplift needs a **randomized pilot**; without one you are estimating a causal effect from observational data, which is a different (harder) course.
- Evaluate with **uplift / Qini curves** on held-out pilot data; never by accuracy.